In [1]:
from langchain.llms import OpenAI
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores.pgvector import PGVector, DistanceStrategy
from langchain.retrievers import ContextualCompressionRetriever
from langchain.chat_models import ChatOpenAI
from langchain.retrievers.document_compressors.chain_extract import LLMChainExtractor
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.callbacks.base import BaseCallbackHandler, AsyncCallbackHandler
from langchain.memory import ConversationBufferMemory
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from IPython.display import Markdown, display, JSON
import re
import json

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [2]:
import dotenv
import os
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from psycopg.conninfo import make_conninfo

dotenv.load_dotenv()
connection_string = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_DATABASE')}"
conn = psycopg2.connect(connection_string)

In [3]:
q = "select * from questions"
questions = []
with conn.cursor() as cur:
    cur.execute(q)
    c = cur.fetchall()
    questions = [q for q in c]

In [3]:
def make_retrievers(key = '350', retrieval_k = 10):
    retrievers = {}
    for collection in ['paper', 'book', 'blog', 'lecture', 'notes']:
        db = PGVector(
            embedding_function=embeddings,
            connection_string=connection_string,
            collection_name=collection + "_" + key,
        )
        retrievers[collection] = db.as_retriever(search_kwargs={"k": retrieval_k})
    base_retriever = MergerRetriever(retrievers=[i for i in retrievers.values()])
    return (retrievers, base_retriever)

r_350 = make_retrievers('350', 10)
r_750 = make_retrievers('750', 10)
r_1500 = make_retrievers('1500', 10)
r_3000 = make_retrievers('3000', 10)

In [8]:
def get_docs(docs, _type = None):
    if _type is None:
        return docs
    else:
        return [i for i in docs if i.metadata['type'] == _type]

all_docs_350 = r_350[1].get_relevant_documents(questions[0][1])
all_docs_750 = r_750[1].get_relevant_documents(questions[0][1])
all_docs_1500 = r_1500[1].get_relevant_documents(questions[0][1])
all_docs_3000 = r_3000[1].get_relevant_documents(questions[0][1])

NameError: name 'questions' is not defined

In [9]:
s = "Consider a competitive market with linear supply and demand curves. Assume the demand curve slopes downward as usual. If the supply curve slopes downward as well, the market will definitely reach an equilibrium price and quantity when supply and demand intersect."
def get_docs(docs, _type = None):
    if _type is None:
        return docs
    else:
        return [i for i in docs if i.metadata['type'] == _type]

all_docs_350 = r_350[1].get_relevant_documents(s)
all_docs_750 = r_750[1].get_relevant_documents(s)
all_docs_1500 = r_1500[1].get_relevant_documents(s)
all_docs_3000 = r_3000[1].get_relevant_documents(s)

In [53]:
def get_doc(addr):
    q = "select * from questions where addr='{}'".format(addr)
    with conn.cursor() as cur:
        cur.execute(q)
        return cur.fetchone()
doc = get_doc('2016f.1.03')
doc_refs_350 = r_350[1].get_relevant_documents(doc[1])
doc_refs_750 = r_750[1].get_relevant_documents(doc[1])
doc_refs_1500 = r_1500[1].get_relevant_documents(doc[1])
doc_refs_3000 = r_3000[1].get_relevant_documents(doc[1])

In [27]:
all_docs_750[0]

Document(page_content='Using his value scale approach, Rothbard (1962) claims to derive the laws of demand and supply as exceptionless theorems.\n\nWhen he says "demand must either increase or remain the same as the price decreases" and "supply must always remain unchanged or increase with an increase in price," he literally means "must" (pp. 106-7; emphasis mine).\n\nBut in his later discussion of labor and land, Rothbard concedes the theoretical possibility of backward bending supply curves (pp. 515-6).\n\nFurthermore, in his treatment of the economics 8 As an anonymous referee pointed out, if the goods are discrete but the monetary unit is continuous, then there will typically be multiple equilibria, a range of prices that clear the market.', metadata={'title': 'The Austrian Search for Realistic Foundations', 'author': 'Bryan Caplan', 'type': 'paper', 'ref': 'Caplan, Bryan. The Austrian Search for Realistic Foundations. April 1999. Southern Economic Journal 65(4), pp.823-838.', 'pag

In [10]:
for i in get_docs(all_docs_1500):
    display(i.metadata)
    display(Markdown(re.sub("\n+", "\n", i.page_content)))
    print("------")


{'title': 'The Meaning ofCompetition',
 'author': 'Friedrich von Hayek',
 'type': 'paper',
 'ref': 'Friedrich von Hayek. The Meaning ofCompetition.',
 'page_number': 3}

2 That the modern theory of competitive equilibrium assumes the situation to exist which a true explanation ought to account for as the effect of the competitive process is best shown by examining the familiar list of conditions found in any modern textbook.
Most of these conditions, incidentally, not only underlie the analysis of "perfect" competition but are equally assumed in the discussion of the various "imperfect" or "monopolistic" markets, which throughout assume certain unrealistic "perfections.
For our immediate purpose, however, the theory of perfect competition will be the most instructive case to examine.
Particularly the assumptions that at all times a uniform price must rule for a given commodity throughout the market and that sellers know the shape of the demand curve.

------


{'title': 'Price Theory and Application',
 'author': 'Jack Hirshleifer, Amihai Glazer, David Hisrhleifer',
 'type': 'book',
 'ref': 'Jack Hirshleifer, Amihai Glazer, David Hisrhleifer. Price Theory and Application.',
 'page_number': 43}

The positive slope of the supply curve indicates that the higher the price, the greater the quantity offered.
In Figure 2.1 market equilibrium occurs at point E where the supply curve and demand curve intersect.
The coordinates of E are the equilibrium quantity Q and equilibrium price P .
To see why this is an equilibrium, consider a market price higher than P for example, P in the diagram.
At price P suppliers want to sell the quantity Q s , while consumers want to buy only Q d .
Since suppliers in aggregate are unable to sell all they want to at price P , at least some of them are likely to offer buyers better terms.
So, as indicated by the downward-pointing arrow, at P there would be downward pressure on price.
Consider next a market price initially lower than P , say P in Figure 2.1.
At such a low price, the quantity Q d that consumers want to buy exceeds the quantity 1
Sometimes it is convenient to interpret quantity on the horizontal axis as a rate per unit time, for instance, thousands of chips per month or per week.

...

So standard monopoly model, where you've got your marginal cost, you've got your demand, and then of course you have your marginal revenue curve, and then you monopolize that marginal revenue for a marginal cost, providing these monopoly profits, and creating these deadweight losses here.
So everyone's seen this model a bunch of times, right?
Okay, all right, so monopolist maximizes PQ minus TC, says marginal revenue plus marginal cost.
All right, so does this make sense in game theoretic terms?
All right, so certainly this is an equilibrium.
This is one equilibrium, but we can imagine all kinds of other equilibria in this monopoly game.
So we can imagine one where consumers refuse to buy at any price higher than marginal cost, and if the consumers won't buy anything at a price higher than marginal cost, what is the monopolist gonna do?
I'll set price.
Yes, we'll set price equal to marginal cost.
So I got nothing to lose then.
My question is, is it possible for the consumer to refuse to buy if the monopolist's set P is greater than a minus?
Well, so we can certainly imagine an equilibrium where this is an equilibrium.
So if you just think of this, what I described is a Nash equilibrium, right?
Because if consumers will not buy anything if the price is greater than marginal cost, then if the monopolist says, charging price equal to marginal cost, if consumers decide they're willing to pay more, does that give them a better deal?

------


{'title': 'Classroom notes',
 'author': 'Bryan Caplan',
 'type': 'notes',
 'ref': '',
 'page_number': 9}

G. Assumption 3: The total demand function for k is continuous in pk for 0<pk<1.
H. In a 2-commodity world (k=2), you can prove the existence of general equilibrium using the Intermediate Value Theorem.
If one market clears, the other has to clear, and if demand is continuous and can be too high or too low, it must at some point be just right.
In a k-commodity world, you can prove the existence of general equilibrium using Brouwer's Fixed Point Theorem.
Basic idea of fixed point theorems: find conditions for functions such that there must be an f(x)=x.
All of our assumptions together conveniently satisfy Brouwer's Fixed Point Theorem, so QED.
J. How do you get to these fixed points?
GE theory usually focuses on the "Walrasian auctioneer" who adjusts price vectors to

------


# Asking questions without document set

In [100]:
import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

messages = [{"role": "system", "content": "You are a PhD student in the Economics department. You are teacher assistant for microeconomics class."}]

def query_llm(query):
    messages.append({"role": "user", "content": query})
    r = openai.ChatCompletion.create(
            model="gpt-4",
            messages=messages)
    messages.append({"role": r["choices"][0]["message"]["role"], "content": r["choices"][0]["message"]["content"]})
    return messages[-1]["content"]

In [104]:
answer = "Let A represent 'minimum wage raises unemployment' and B represent 'AER article finds that minimum wage does not raise unemployment'. We need to find P(A) using P(A|B), P(~B|A), and P(~B|~A). P(~B|A) = 1 - P(B|A). Using Bayes' Law, we can calculate P(A) for both yourself and your friend: For you: .9 = (.25P(A))/(.25P(A)+.75(1-P(A))), which implies P(A) = 0.964. For your friend: .45=(.25P(A))/(.25P(A)+.75(1-P(A))) , which implies P(A) = 0.771. By observation, .964 is not equal to 2 * 0.771. The question's statement is false. It's important to note that the text mistakenly gives the probabilities for when the article finds the minimum wage raises unemployment, while the article actually finds the minimum wage does not raise unemployment. The statement's error implies that our confidence in the minimum wage causing unemployment increases despite seeing conflicting evidence."
messages = [{"role": "system", "content": "You are a PhD student in the Economics department. You are teacher assistant for microeconomics class. Your school follows Austrian economics."}]
display(Markdown(query_llm(f"Answer this question:  {questions[0][1]}")))
print("-----")
display(Markdown(query_llm(f"analyze this answer for the question: {answer}")))

False. The relation between the two posterior probabilities cannot be directly inferred from the given likelihoods and priors only. We can understand the resulting posterior probability using Bayes' theorem which integrates prior beliefs (prior probabilities) with new evidence (the likelihood). According to the theorem, the posterior probability is a function of both the prior probability and the likelihood. 

This means that even if two persons agree on the likelihoods, if their prior probabilities differ, their posterior probabilities can be vastly different. It's not necessarily a simple doubling relationship. Determining the exact prior probabilities that you and your friend used would require more information and mathematical computation beyond what is provided. So, it can't be definitively stated that your prior probability is exactly double your friend's.

-----


Your analysis is mostly correct, but there seems to be a confusion. The probabilities that were provided are not P(B|A) and P(~B|~A) as you mentioned, but instead the probabilities P(B|A) and P(B|~A). With B denoting 'article finds minimum wage raises unemployment' and A denoting 'minimum wage raises unemployment', the provided probabilities represent the likelihood of the article finding an effect given that there is an effect (P(B|A)) and the likelihood of the article finding an effect given that there is no effect (P(B|~A)). 

However, the analysis following this misunderstanding is correct. By correctly applying Bayes' rule, you calculated the individual prior beliefs about the impact of the minimum wage on unemployment (P(A)). Indeed, your prior probability (.964) is not exactly double the prior probability of your friend (.771), therefore the statement in the question is false. But, as you noted, the probabilities used here seem to contradict the finding of the article.

In [113]:
# biased as austrian
messages = [{"role": "system", "content": "You are a PhD student in the Economics department. You are assistant for microeconomics class. you are biased towards austrian economics."}]
display(Markdown(query_llm(f"Answer this question:  {questions[0][1]}")))
print("-----")
display(Markdown(query_llm(f"analyze this answer for the question, explain in details what you disagree with it: {answer}")))

False

The statement does not hold true and the explanation for this lies in the concept of Bayesian statistics and how it updates the prior belief.

According to Bayes' theorem: P(A|B) = [P(B|A)*P(A)]/P(B)

where P(A|B) is the probability of event A given event B is true,
P(B|A) is the probability of event B given event A is true,
P(A) is the prior probability of event A,
and P(B) is the prior probability of event B.

In the given case, the probability of the article's findings given that the minimum wage really does raise unemployment (P(B|A)) is 0.75. The probability of the article's findings given that the minimum wage really does not raise unemployment (P(B|not A)) is 0.25. P(A|B) and P(A) are the post and prior beliefs about whether the minimum wage increases unemployment.

The disagreement between your final estimate and your friend’s does not automatically mean that your prior belief must be double your friend's. The discrepancy between beliefs lies in the different levels of confidence each of you place in the validity of the AER article’s findings, which is related to each individual's interpretation of the report's potential for bias, amongst other subjective factors. Without knowing these variables, it is impossible to state that your prior belief must be double your friend’s prior belief. This situation is a reflection of the subjectivity inherent in Bayesian interpretation.

-----


While the use of Bayes' law to update prior beliefs about whether minimum wage raises unemployment (event A) based on the AER article (event B) is correctly applied in this explanation, there remains an issue with the interpretation of the problem. 

Crucially, the explanation assumes without justification that the probability of the study finding the opposite of the actual underlying reality (P(~B|A) and P(~B|~A)) are directly complementary to the respective given probabilities (1 - P(B|A) and 1 - P(B|~A)). The implicit assertion here is that all studies either correctly affirm or incorrectly deny the true state of the world without any possibility of inconclusiveness or investigation of other aspects. 

Moreover, the explanation doesn't account for the Austrian Economics bias of the PhD student. That bias could potentially influence how the data is read, interpreted, and thus the initially assigned probability values.

Further, although the explanation correctly points out that the answer refers to probabilities for when the article finds minimum wage raises unemployment, the article in question actually finds the contrary, the probability values (0.75 and 0.25) assigned might not perfectly translate in the opposite direction.

This process of applying Bayes' Law is mathematically accurate. Still, the interpretation and expected outcome might not hold accurate to real-life situations due to the many complexities, like biases and non-binary nature of research results. Thus, we cannot definitively claim the prior probability that the minimum wage increases unemployment to be exactly double for the PhD student.


In [112]:
messages = [{"role": "system", "content": "You are a PhD student in the Economics department. You are teacher assistant for microeconomics class."}]
display(Markdown(query_llm(f"Answer this question, assume that my friend and I increase our belief in minimum wage raises unemployment after reading the article:\n\n{questions[0][1]}")))
display(Markdown(query_llm(f"analyze this answer for the question, explain in details what you disagree with it: {answer}")))

False. The probability calculations explained in the problem pertain to Bayesian updating, which involves adjusting your prior beliefs based upon new evidence. It is a method of applying Bayes' theorem.

The difference in the final probability assessments between you and your friend could be accounted by the difference in each of your prior probabilities. However, saying that your prior probability is exactly double your friend's is too specific a claim. Bayesian updating not just depends on the prior probabilities (your initial belief about the minimum wage raising unemployment), but also on how much weight you give to the new piece of evidence (the article in this case). It's plausible that you review the piece of evidence with a higher weight compared to your friend, hence your posterior probability could higher.

Simply put, the difference in final probabilities depends on both the difference in prior probabilities and the different weight given to the new evidence. Therefore, it is incorrect to make a direct correlation that your prior probability that the minimum wage increases unemployment must be exactly double your friend's prior probability.

The main disagreement with the provided answer lies in the interpretation of the results and the explanation of Bayes' Law. The answer correctly applies the law but fails to highlight the key conceptual aspects and interpret the results appropriately.

1. The calculations of the prior probabilities, P(A) for both you (.964) and your friend (.771), are correctly calculated using Bayes' Theorem. However, this does not directly combat the original question.

2. The prompt requested whether your prior probability that minimum wage raises unemployment should be exactly double your friend's prior, not a calculation of what those priors might be given the observed posterior probabilities. 

3. By merely calculating and comparing these probabilities, the answer inappropriately compares the calculated prior probabilities after seeing the evidence to decide on the original question. This overlooks the influence of how each individual might weigh the new evidence, which could vary the posterior probabilities without strictly doubling the prior probabilities.

4. The answer ends by pointing out the texts mistake - presenting probabilities for when the article supports minimum wage increasing unemployment when the article findings are opposite. However, this point is not directly related to the validity of the original question. 

So, while the answer invokes Bayes' Law correctly and suggests a disagreement with the statement, it misinterprets what is being disputed and does not give a detailed conceptual explanation.

# Asking questions with Document Sets

In [6]:
import sys
class MyCustomHandlerOne(BaseCallbackHandler):
    def on_llm_new_token(self, token, **kwargs):
        print(token, end="")
        sys.stdout.flush()

    def on_llm_end(self, outputs, **kwargs):
        print("\n\n")

llm = ChatOpenAI(temperature = 0.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=False,
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))

{'question': "You and a friend both read the same article in the AER finding that the minimum wage does not increase unemployment. You both agree that P(article finds minimum wage raises unemployment | the minimum wage really does raise unemployment)=.75, and P(article finds minimum wage raises unemployment | the minimum wage really does not raise unemployment)=.25. But the two of you DISAGREE in your final estimates: your P(minimum wage raises unemployment| article's findings)=.9, while your friend sets the same probability at .45. True, False, and Explain: Your prior probability that the minimum wage increases unemployment must be exactly double your friend's prior probability that the minimum wage increases unemployment.", 'context': '33 NEED2EARN 0 = \'Can make living with one wage 0.75 0.87 earner\'; 1 = \'Agree that need two wage earners Over the next five years, do you think the average American\'s standard of living will rise, or fall, or stay about the same?\n\nFor example, th

True. This is a question of Bayesian updating. Given that you both agree on the likelihood of the article's findings given whether the minimum wage raises unemployment or not, the only way for you to arrive at different posterior probabilities is if you started with different prior probabilities. Specifically, your prior probability that the minimum wage increases unemployment must be higher than your friend's. In fact, given the posterior probabilities you each arrived at, your prior must be exactly double your friend's prior.

In [126]:
llm = ChatOpenAI(temperature = 0.2, model = 'gpt-3.5-turbo-16k', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=False,
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))

Prompt=None
False. The prior probability that the minimum wage increases unemployment does not have to be exactly double your friend's prior probability. The prior probabilities can vary depending on individual beliefs and interpretations of the evidence.




False. The prior probability that the minimum wage increases unemployment does not have to be exactly double your friend's prior probability. The prior probabilities can vary depending on individual beliefs and interpretations of the evidence.

In [40]:
stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics.

This is information from your references: {context}

Answer considering only the reference material: {question}

Lecture and expand concepts required to answer.
"""
# stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics. Use the following pieces of context to answer the question at the end.

# {context}

# Question: {question}
# Helpful Answer:"""
PROMPT = PromptTemplate(
    template=stuff_prompt_template, input_variables=["context", "question"]
)
llm = ChatOpenAI(temperature = 0.2, model = 'gpt-3.5-turbo-16k', callbacks=[MyCustomHandlerOne()], streaming=True)
# llm = ChatOpenAI(temperature = 0.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=True,
    chain_type_kwargs={"prompt": PROMPT},
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))



> Entering new RetrievalQA chain...
{'question': "You and a friend both read the same article in the AER finding that the minimum wage does not increase unemployment. You both agree that P(article finds minimum wage raises unemployment | the minimum wage really does raise unemployment)=.75, and P(article finds minimum wage raises unemployment | the minimum wage really does not raise unemployment)=.25. But the two of you DISAGREE in your final estimates: your P(minimum wage raises unemployment| article's findings)=.9, while your friend sets the same probability at .45. True, False, and Explain: Your prior probability that the minimum wage increases unemployment must be exactly double your friend's prior probability that the minimum wage increases unemployment.", 'context': '33 NEED2EARN 0 = \'Can make living with one wage 0.75 0.87 earner\'; 1 = \'Agree that need two wage earners Over the next five years, do you think the average American\'s standard of living will rise, or fall, or s

To answer this question, we need to understand the concept of prior probability and how it relates to the given information. Prior probability refers to the initial belief or probability assigned to an event before any evidence or information is considered.

In this case, both you and your friend read the same article in the AER (American Economic Review) that finds the minimum wage does not increase unemployment. You both agree on the conditional probabilities: P(article finds minimum wage raises unemployment | the minimum wage really does raise unemployment) = 0.75 and P(article finds minimum wage raises unemployment | the minimum wage really does not raise unemployment) = 0.25.

However, you and your friend disagree in your final estimates: your P(minimum wage raises unemployment | article's findings) = 0.9, while your friend sets the same probability at 0.45.

Based on this information, it is not necessarily true that your prior probability that the minimum wage increases unemployment must be exactly double your friend's prior probability. The prior probability is the initial belief before considering any evidence. The given information does not provide any details about your or your friend's prior probabilities. Therefore, we cannot determine if your prior probability is exactly double your friend's prior probability based on the information provided.

### The exact answer with 3.5, is this a fluke?

In [18]:
stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics.

This is information from your references: {context}

Answer considering only the reference material: {question}

Lecture and expand concepts required to answer.
"""
# stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics. Use the following pieces of context to answer the question at the end.

# {context}

# Question: {question}
# Helpful Answer:"""
PROMPT = PromptTemplate(
    template=stuff_prompt_template, input_variables=["context", "question"]
)
llm = ChatOpenAI(temperature = 0.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=True,
    chain_type_kwargs={"prompt": PROMPT},
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))



> Entering new RetrievalQA chain...
{'question': "You and a friend both read the same article in the AER finding that the minimum wage does not increase unemployment. You both agree that P(article finds minimum wage raises unemployment | the minimum wage really does raise unemployment)=.75, and P(article finds minimum wage raises unemployment | the minimum wage really does not raise unemployment)=.25. But the two of you DISAGREE in your final estimates: your P(minimum wage raises unemployment| article's findings)=.9, while your friend sets the same probability at .45. True, False, and Explain: Your prior probability that the minimum wage increases unemployment must be exactly double your friend's prior probability that the minimum wage increases unemployment.", 'context': '33 NEED2EARN 0 = \'Can make living with one wage 0.75 0.87 earner\'; 1 = \'Agree that need two wage earners Over the next five years, do you think the average American\'s standard of living will rise, or fall, or s

The question is asking about the application of Bayes' theorem in the context of beliefs about the impact of minimum wage on unemployment. Bayes' theorem is a mathematical formula used in probability theory and statistics to calculate conditional probability. In this context, it is used to update the probability of a hypothesis (in this case, that the minimum wage increases unemployment) based on new evidence (the article's findings).

The formula for Bayes' theorem is:

P(A|B) = P(B|A) * P(A) / P(B)

Where:
- P(A|B) is the probability of event A given event B is true
- P(B|A) is the probability of event B given event A is true
- P(A) and P(B) are the probabilities of events A and B respectively

In this scenario:
- A is the event "minimum wage raises unemployment"
- B is the event "article's findings"
- P(A|B) is the updated belief about the impact of minimum wage on unemployment after reading the article
- P(B|A) is the belief about the likelihood of the article's findings given the true impact of minimum wage on unemployment
- P(A) is the prior belief about the impact of minimum wage on unemployment before reading the article
- P(B) is the belief about the likelihood of the article's findings

The question states that you and your friend agree on P(B|A) and P(B|~A), but disagree on P(A|B). According to Bayes' theorem, this implies that you and your friend must have different prior probabilities P(A), because the only other factors in the equation are agreed upon.

Therefore, the statement is true: your prior probability that the minimum wage increases unemployment must be exactly double your friend's prior probability that the minimum wage increases unemployment. This is because your updated belief P(A|B) is exactly double your friend's updated belief, and the other factors in Bayes' theorem are the same for both of you.

### GPT4 insists in getting it wrong almost every time

In [19]:
stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics.

Use only this information: {context}

Answer considering only the reference material: {question}

Lecture and expand concepts required to answer.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
"""
# stuff_prompt_template = """You are student in the Economics PhD, assistant for microeonomics. Use the following pieces of context to answer the question at the end.

# {context}

# Question: {question}
# Helpful Answer:"""
PROMPT = PromptTemplate(
    template=stuff_prompt_template, input_variables=["context", "question"]
)
llm = ChatOpenAI(temperature = 0.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=r_350[1],
    return_source_documents=True,
    verbose=True,
    chain_type_kwargs={"prompt": PROMPT},
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))



> Entering new RetrievalQA chain...
{'question': "You and a friend both read the same article in the AER finding that the minimum wage does not increase unemployment. You both agree that P(article finds minimum wage raises unemployment | the minimum wage really does raise unemployment)=.75, and P(article finds minimum wage raises unemployment | the minimum wage really does not raise unemployment)=.25. But the two of you DISAGREE in your final estimates: your P(minimum wage raises unemployment| article's findings)=.9, while your friend sets the same probability at .45. True, False, and Explain: Your prior probability that the minimum wage increases unemployment must be exactly double your friend's prior probability that the minimum wage increases unemployment.", 'context': '33 NEED2EARN 0 = \'Can make living with one wage 0.75 0.87 earner\'; 1 = \'Agree that need two wage earners Over the next five years, do you think the average American\'s standard of living will rise, or fall, or s

True. This is a direct application of Bayes' theorem, which is a fundamental concept in probability theory and statistics. The theorem describes the probability of an event based on prior knowledge of conditions that might be related to the event. In this case, the event is "the minimum wage increases unemployment" and the condition is "the article's findings". 

According to Bayes' theorem, the posterior probability (your final estimate) is proportional to the prior probability (your initial belief) times the likelihood (the probability of the evidence given the hypothesis). 

In mathematical terms, P(A|B) = P(B|A) * P(A) / P(B), where:
- P(A|B) is the posterior probability (your final estimate)
- P(B|A) is the likelihood (the probability of the evidence given the hypothesis)
- P(A) is the prior probability (your initial belief)
- P(B) is the evidence (the article's findings)

Given that you both agree on the likelihoods, the only way for your posterior probabilities to differ is if your prior probabilities differ. Specifically, if your posterior probability is double your friend's, then your prior probability must also be double your friend's.

In [ ]:
mr_prompt_template = """You are student in the Economics PhD, assistant for microeonomics. Use the following portion of a long document to see if any of the text is relevant to answer the question.
When input text is relevant, return lecture about the relation between question and input. Otherwise reply "No comment".

Document: {context}

Question: {question}

Comment:
"""

mr_combine_prompt_template = """You are student in the Economics PhD, assistant for microeonomics.
Given the following extracted parts of a long document and a question, create a final answer.
Lecture, give examples and present and explain concepts of microeconomics contained in your final answer.

SOURCES:

QUESTION: {question}
=========
{summaries}
=========
FINAL ANSWER:"""

QUESTION_PROMPT = PromptTemplate(
    template=mr_prompt_template, input_variables=["context", "question"]
)
COMBINE_PROMPT = PromptTemplate(
    template=mr_combine_prompt_template, input_variables=["summaries", "question"]
)
llm = ChatOpenAI(temperature = 1.2, model = 'gpt-4', callbacks=[MyCustomHandlerOne()], streaming=True)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="map_reduce",
    retriever=r_1500[0]["lecture"],
    return_source_documents=True,
    verbose=False,
    chain_type_kwargs={"question_prompt": QUESTION_PROMPT, "combine_prompt": COMBINE_PROMPT},
)
retval = qa_chain({"query": questions[0][1]})
display(Markdown(retval["result"]))